In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION


Purpose:
    Develop the final multi-label disease classification head using the
    locked Experiment E (Seed 42) bilateral fused representation.

Locked Upstream Configuration:
    • Experiment: E — Adaptive Bilateral Fusion
    • Seed: 42
    • Best Epoch: 5
    • Val Macro-F1: 0.561936
    • Val Micro-F1: 0.535966
    • Exact Match: 0.175799
    • Locked artifact: stage2_experiment_e_locked_seed42.pt

Strict Constraints:
    • Experiment E is FROZEN.
    • Cross-Eye representations are FROZEN.
    • Adaptive bilateral fusion is FROZEN.
    • No changes to the upstream backbone, disease attention,
      cross-eye attention, or fusion mechanism.
    • Only the downstream multi-label classification module will be optimized.

Classification Objective:
    Predict the 8 disease labels simultaneously from the locked fused
    bilateral representation.

Optimization Goal:
    Maximize multi-label diagnostic performance, with particular emphasis on:
    • Macro-F1
    • Micro-F1
    • Per-disease F1
    • Exact Match
    • Robust handling of class imbalance

Experimental Strategy:
    Start with strong, established multi-label classification approaches
    rather than weak baseline architectures or arbitrary architectural changes.

    Initial focus:
    • Strong classification head
    • Appropriate normalization and regularization
    • Class-imbalance-aware loss
    • Multi-label-specific loss functions
    • Validation-based threshold optimization
    • Per-disease performance analysis

Important:
    The classification module must consume the locked Experiment E
    representation directly. Upstream representations must not be retrained
    or modified during classifier development.



In [14]:
# =============================================================================
# QCDP-BiFormer — MODULE 9
# MULTI-LABEL CLASSIFICATION
# CELL 1 — ENVIRONMENT + LOCKED RESOURCES
# =============================================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount('/content/drive')

ROOT = Path("/content/drive/My Drive/Eye Disease/Dataset")

assert ROOT.exists(), f"Dataset root not found: {ROOT}"

print("=" * 80)
print("QCDP-BiFormer — MULTI-LABEL CLASSIFICATION")
print("=" * 80)
print(f"Dataset root: {ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior for evaluation/reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"\nGlobal seed: {SEED}")

# -----------------------------------------------------------------------------
# 3. LOCKED EXPERIMENT
# -----------------------------------------------------------------------------

LOCKED_EXPERIMENT = "experiment_e"
LOCKED_SEED = 42

LOCKED_ARTIFACT = ROOT / "stage2_experiment_e_locked_seed42.pt"

assert LOCKED_ARTIFACT.exists(), (
    f"Locked Experiment E artifact not found:\n{LOCKED_ARTIFACT}"
)

print("\nLocked configuration:")
print(f"  Experiment : {LOCKED_EXPERIMENT}")
print(f"  Seed       : {LOCKED_SEED}")
print(f"  Artifact   : {LOCKED_ARTIFACT}")

# -----------------------------------------------------------------------------
# 4. DATASET / LABEL DEFINITIONS
# -----------------------------------------------------------------------------

TRAIN_DF_PATH = ROOT / "stage2_train_df.csv"
VAL_DF_PATH   = ROOT / "val_patient_df.csv"

assert TRAIN_DF_PATH.exists(), f"Missing: {TRAIN_DF_PATH}"
assert VAL_DF_PATH.exists(), f"Missing: {VAL_DF_PATH}"

train_df = pd.read_csv(TRAIN_DF_PATH)
val_df   = pd.read_csv(VAL_DF_PATH)

# Fixed disease ordering used throughout QCDP-BiFormer
DISEASE_NAMES = [
    "N",  # Normal
    "D",  # Diabetic Retinopathy
    "G",  # Glaucoma
    "C",  # Cataract
    "A",  # AMD
    "H",  # Hypertension
    "M",  # Myopia
    "O",  # Other
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\nDataset:")
print(f"  Train samples : {len(train_df)}")
print(f"  Val samples   : {len(val_df)}")
print(f"  Classes       : {NUM_CLASSES}")
print(f"  Label order   : {DISEASE_NAMES}")

# -----------------------------------------------------------------------------
# 5. EXISTING LOCKED / UPSTREAM RESOURCES
# -----------------------------------------------------------------------------

RESOURCE_FILES = {
    "stage1_train_df": ROOT / "stage1_train_df.csv",
    "stage2_train_df": ROOT / "stage2_train_df.csv",
    "stage1_quality_scores": ROOT / "stage1_quality_scores.csv",

    "disease_aware_features": ROOT / "disease_aware_features.pt",
    "disease_attention_weights": ROOT / "disease_attention_weights.pt",
    "disease_prototypes": ROOT / "disease_prototypes.pt",

    "cross_eye_train": ROOT / "stage2_train_cross_eye_features_final.pt",
    "cross_eye_val": ROOT / "stage2_val_cross_eye_features_final.pt",

    "locked_experiment_e": LOCKED_ARTIFACT,
}

print("\nUpstream resources:")
for name, path in RESOURCE_FILES.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"  [{status:>9}] {name}: {path.name}")

# -----------------------------------------------------------------------------
# 6. LOAD LOCKED EXPERIMENT E ARTIFACT
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Loading locked Experiment E artifact...")
print("-" * 80)

locked_artifact = torch.load(
    LOCKED_ARTIFACT,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(locked_artifact)}")

# Inspect structure without modifying anything
if isinstance(locked_artifact, dict):

    print("\nArtifact contents:")
    for key, value in locked_artifact.items():

        if torch.is_tensor(value):
            print(
                f"  {key}: Tensor "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

        elif isinstance(value, dict):
            print(
                f"  {key}: dict "
                f"({len(value)} entries)"
            )

        elif isinstance(value, (list, tuple)):
            print(
                f"  {key}: {type(value).__name__} "
                f"({len(value)} entries)"
            )

        else:
            print(
                f"  {key}: "
                f"{type(value).__name__} = {value}"
            )

# -----------------------------------------------------------------------------
# 7. LOCKED REFERENCE METRICS
# -----------------------------------------------------------------------------

LOCKED_REFERENCE = {
    "best_epoch": 5,
    "val_loss": 0.791240,
    "micro_f1": 0.535966,
    "macro_f1": 0.561936,
    "exact_match": 0.175799,
}

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E / SEED 42 REFERENCE")
print("-" * 80)

for metric, value in LOCKED_REFERENCE.items():
    print(f"{metric:>15}: {value}")

# -----------------------------------------------------------------------------
# 8. GLOBAL CLASSIFICATION CONFIGURATION
# -----------------------------------------------------------------------------

FEATURE_DIM = 768
NUM_CLASSES = 8

# We will NOT modify these upstream representations.
UPSTREAM_FROZEN = True

print("\n" + "=" * 80)
print("MODULE 9 INITIALIZATION COMPLETE")
print("=" * 80)

print(f"""
Locked Experiment       : E
Locked Seed             : 42
Feature dimension       : {FEATURE_DIM}
Number of diseases      : {NUM_CLASSES}
Disease ordering        : {DISEASE_NAMES}

Upstream fusion         : FROZEN
Cross-Eye representations: FROZEN
Adaptive fusion         : FROZEN
Mean-fusion replacement : NOT ALLOWED

""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QCDP-BiFormer — MULTI-LABEL CLASSIFICATION
Dataset root: /content/drive/My Drive/Eye Disease/Dataset
PyTorch version: 2.11.0+cpu
CUDA available: False

Global seed: 42

Locked configuration:
  Experiment : experiment_e
  Seed       : 42
  Artifact   : /content/drive/My Drive/Eye Disease/Dataset/stage2_experiment_e_locked_seed42.pt

Dataset:
  Train samples : 2141
  Val samples   : 504
  Classes       : 8
  Label order   : ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Upstream resources:
  [    FOUND] stage1_train_df: stage1_train_df.csv
  [    FOUND] stage2_train_df: stage2_train_df.csv
  [    FOUND] stage1_quality_scores: stage1_quality_scores.csv
  [    FOUND] disease_aware_features: disease_aware_features.pt
  [    FOUND] disease_attention_weights: disease_attention_weights.pt
  [    FOUND] disease_prototypes: disease_prototypes.pt
  [    FOUND] cross_eye_trai

CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS


Objective:
    Load the officially exported Experiment E / Seed 42 fused representations
    and verify their integrity before designing and training the multi-label
    classification head.

OFFICIALLY LOCKED UPSTREAM:
    Experiment E — Strict Class-wise Adaptive Fusion
    Seed: 42

Representation:
    Train: [2141, 8, 768]
    Val  : [438, 8, 768]

Interpretation:
    Each patient has 8 disease-specific fused representations, with each
    disease receiving its own 768-dimensional representation.

Disease order:
    ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Disease mapping:
    N = Normal
    D = Diabetic Retinopathy
    G = Glaucoma
    C = Cataract
    A = Age-related Macular Degeneration
    H = Hypertension
    M = Myopia
    O = Other

Validation cohort:
    The 438-patient bilateral cohort is used because it is the exact cohort
    on which the locked Experiment E representation was generated.

STRICT UPSTREAM LOCK:
    • No fusion retraining
    • No feature modification
    • No new patient filtering
    • No new train/validation split
    • No mean-fusion replacement
    • No changes to Experiment E

This cell performs verification only.

Next:
    Determine the most appropriate classification-head input strategy for
    the disease-specific [8 × 768] representation before beginning
    optimization experiments.


In [15]:
# =============================================================================
# QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION
# CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 80)
print("CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/My Drive/Eye Disease/Dataset"
)

FUSED_ARTIFACT_PATH = (
    ROOT / "stage2_experiment_e_fused_representations_seed42.pt"
)

TRAIN_LABEL_PATH = (
    ROOT / "stage2_train_bilateral_metadata.csv"
)

VAL_LABEL_PATH = (
    ROOT / "stage2_val_bilateral_metadata.csv"
)

assert FUSED_ARTIFACT_PATH.exists(), (
    f"Missing fused representation artifact:\n{FUSED_ARTIFACT_PATH}"
)

assert TRAIN_LABEL_PATH.exists(), (
    f"Missing train bilateral metadata:\n{TRAIN_LABEL_PATH}"
)

assert VAL_LABEL_PATH.exists(), (
    f"Missing validation bilateral metadata:\n{VAL_LABEL_PATH}"
)

print(f"\nDataset root: {ROOT}")
print(f"Fused artifact: {FUSED_ARTIFACT_PATH.name}")


# =============================================================================
# 2. LOAD ARTIFACT
# =============================================================================

print("\n" + "-" * 80)
print("LOADING LOCKED FUSED REPRESENTATIONS")
print("-" * 80)

fused_artifact = torch.load(
    FUSED_ARTIFACT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(fused_artifact)}")


# =============================================================================
# 3. VERIFY LOCKED PROVENANCE
# =============================================================================

print("\n" + "-" * 80)
print("VERIFYING LOCKED PROVENANCE")
print("-" * 80)

print(
    f"Experiment     : "
    f"{fused_artifact.get('experiment')}"
)

print(
    f"Experiment key : "
    f"{fused_artifact.get('experiment_key')}"
)

print(
    f"Seed           : "
    f"{fused_artifact.get('seed')}"
)

print(
    f"Status         : "
    f"{fused_artifact.get('status')}"
)

assert fused_artifact["experiment_key"] == "experiment_e"
assert fused_artifact["seed"] == 42
assert fused_artifact["status"] == "LOCKED"

print("\n✓ Experiment E verified.")
print("✓ Seed 42 verified.")
print("✓ Artifact status = LOCKED.")


# =============================================================================
# 4. DISEASE ORDER
# =============================================================================

DISEASE_NAMES = [
    "N",
    "D",
    "G",
    "C",
    "A",
    "H",
    "M",
    "O",
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\n" + "-" * 80)
print("DISEASE ORDER")
print("-" * 80)

print(DISEASE_NAMES)

artifact_labels = list(
    fused_artifact["label_columns"]
)

print(
    f"\nArtifact label order: {artifact_labels}"
)

assert artifact_labels == DISEASE_NAMES, (
    "Artifact disease ordering does not match Module 9 ordering."
)

print("✓ Disease ordering verified.")


# =============================================================================
# 5. EXTRACT FUSED FEATURES
# =============================================================================

print("\n" + "-" * 80)
print("EXTRACTING FUSED FEATURES")
print("-" * 80)

X_train = fused_artifact[
    "train_classwise_features"
].float()

X_val = fused_artifact[
    "val_classwise_features"
].float()


# -----------------------------------------------------------------------------
# IMPORTANT:
# The fused artifact stores patient IDs as 0-D PyTorch tensors:
#     tensor(1), tensor(4), ...
#
# The CSV stores them as integers:
#     1, 4, ...
#
# Normalize them here so the remainder of the notebook works with ordinary
# Python integer patient IDs.
# -----------------------------------------------------------------------------

def normalize_patient_ids(ids):

    normalized = []

    for x in ids:

        if torch.is_tensor(x):
            normalized.append(int(x.item()))

        else:
            normalized.append(int(x))

    return normalized


train_patient_ids = normalize_patient_ids(
    fused_artifact["train_patient_ids"]
)

val_patient_ids = normalize_patient_ids(
    fused_artifact["val_patient_ids"]
)

print(
    f"Train fused representation : "
    f"{tuple(X_train.shape)}"
)

print(
    f"Val fused representation   : "
    f"{tuple(X_val.shape)}"
)

print(
    f"Train patient IDs          : "
    f"{len(train_patient_ids)}"
)

print(
    f"Val patient IDs            : "
    f"{len(val_patient_ids)}"
)

print(
    f"\nFirst 10 normalized train IDs:"
)

print(train_patient_ids[:10])


# =============================================================================
# 6. STRICT SHAPE VERIFICATION
# =============================================================================

assert X_train.ndim == 3
assert X_val.ndim == 3

assert X_train.shape[1] == NUM_CLASSES
assert X_val.shape[1] == NUM_CLASSES

assert X_train.shape[2] == 768
assert X_val.shape[2] == 768

assert X_train.shape[0] == len(train_patient_ids)
assert X_val.shape[0] == len(val_patient_ids)

print("\n✓ Train shape = [2141, 8, 768]")
print("✓ Validation shape = [438, 8, 768]")
print("✓ Eight disease-specific representations confirmed.")
print("✓ Representation dimension = 768.")


# =============================================================================
# 7. CHECK FINITE VALUES
# =============================================================================

print("\n" + "-" * 80)
print("NUMERICAL INTEGRITY")
print("-" * 80)

train_nan = torch.isnan(X_train).sum().item()
train_inf = torch.isinf(X_train).sum().item()

val_nan = torch.isnan(X_val).sum().item()
val_inf = torch.isinf(X_val).sum().item()

print(f"Train NaN values : {train_nan}")
print(f"Train Inf values : {train_inf}")
print(f"Val NaN values   : {val_nan}")
print(f"Val Inf values   : {val_inf}")

assert train_nan == 0
assert train_inf == 0
assert val_nan == 0
assert val_inf == 0

print("\n✓ No NaN values.")
print("✓ No infinite values.")


# =============================================================================
# 8. REPRESENTATION STATISTICS
# =============================================================================

print("\n" + "-" * 80)
print("REPRESENTATION STATISTICS")
print("-" * 80)

print(
    f"Train mean : {X_train.mean().item():.6f}"
)

print(
    f"Train std  : {X_train.std().item():.6f}"
)

print(
    f"Train min  : {X_train.min().item():.6f}"
)

print(
    f"Train max  : {X_train.max().item():.6f}"
)

print()

print(
    f"Val mean   : {X_val.mean().item():.6f}"
)

print(
    f"Val std    : {X_val.std().item():.6f}"
)

print(
    f"Val min    : {X_val.min().item():.6f}"
)

print(
    f"Val max    : {X_val.max().item():.6f}"
)


# =============================================================================
# 9. LOAD BILATERAL LABEL DATA
# =============================================================================

print("\n" + "-" * 80)
print("LOADING BILATERAL LABEL DATA")
print("-" * 80)

train_df = pd.read_csv(
    TRAIN_LABEL_PATH
)

val_df = pd.read_csv(
    VAL_LABEL_PATH
)

print(
    f"Train dataframe: {train_df.shape}"
)

print(
    f"Validation dataframe: {val_df.shape}"
)


# =============================================================================
# 10. VERIFY LABEL COLUMNS
# =============================================================================

missing_train = [
    disease for disease in DISEASE_NAMES
    if disease not in train_df.columns
]

missing_val = [
    disease for disease in DISEASE_NAMES
    if disease not in val_df.columns
]

assert not missing_train, (
    f"Missing train labels: {missing_train}"
)

assert not missing_val, (
    f"Missing validation labels: {missing_val}"
)

print("\n✓ All eight disease labels present.")


# =============================================================================
# 11. VERIFY PATIENT COUNTS
# =============================================================================

assert len(train_df) == X_train.shape[0]
assert len(val_df) == X_val.shape[0]

print("\n" + "-" * 80)
print("PATIENT COUNT ALIGNMENT")
print("-" * 80)

print(
    f"Train features : {len(X_train)}"
)

print(
    f"Train labels   : {len(train_df)}"
)

print(
    f"Val features   : {len(X_val)}"
)

print(
    f"Val labels     : {len(val_df)}"
)

print("\n✓ Train counts aligned.")
print("✓ Validation counts aligned.")


# =============================================================================
# 12. VERIFY PATIENT-ID ALIGNMENT
# =============================================================================

print("\n" + "-" * 80)
print("PATIENT-ID ALIGNMENT")
print("-" * 80)

possible_id_columns = [
    "patient_id",
    "Patient_ID",
    "patientID",
    "PatientID",
    "patient",
    "ID",
    "id",
]

train_id_candidates = [
    c for c in possible_id_columns
    if c in train_df.columns
]

val_id_candidates = [
    c for c in possible_id_columns
    if c in val_df.columns
]

print(
    f"Train ID candidates: {train_id_candidates}"
)

print(
    f"Val ID candidates  : {val_id_candidates}"
)

assert len(train_id_candidates) == 1, (
    "Could not uniquely identify train patient-ID column."
)

assert len(val_id_candidates) == 1, (
    "Could not uniquely identify validation patient-ID column."
)

train_id_col = train_id_candidates[0]
val_id_col = val_id_candidates[0]

train_df_ids = [
    int(x)
    for x in train_df[train_id_col].tolist()
]

val_df_ids = [
    int(x)
    for x in val_df[val_id_col].tolist()
]

assert len(train_df_ids) == len(set(train_df_ids))
assert len(val_df_ids) == len(set(val_df_ids))

train_feature_set = set(train_patient_ids)
train_label_set = set(train_df_ids)

val_feature_set = set(val_patient_ids)
val_label_set = set(val_df_ids)

assert train_feature_set == train_label_set, (
    "Train patient IDs do not match exactly."
)

assert val_feature_set == val_label_set, (
    "Validation patient IDs do not match exactly."
)

print("\n✓ Train patient IDs match exactly.")
print("✓ Validation patient IDs match exactly.")


# =============================================================================
# 13. BUILD LABEL MATRICES
# =============================================================================

print("\n" + "-" * 80)
print("BUILDING MULTI-LABEL TARGET MATRICES")
print("-" * 80)

# Reorder dataframes explicitly according to the fused-feature patient order.

train_lookup = train_df.set_index(train_id_col)

val_lookup = val_df.set_index(val_id_col)

train_ordered_df = train_lookup.loc[
    train_patient_ids
].reset_index()

val_ordered_df = val_lookup.loc[
    val_patient_ids
].reset_index()

Y_train = torch.tensor(
    train_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

Y_val = torch.tensor(
    val_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

print(
    f"Y_train shape: {tuple(Y_train.shape)}"
)

print(
    f"Y_val shape  : {tuple(Y_val.shape)}"
)

assert Y_train.shape == (
    X_train.shape[0],
    NUM_CLASSES
)

assert Y_val.shape == (
    X_val.shape[0],
    NUM_CLASSES
)

print("\n✓ Feature → patient → label ordering verified.")


# =============================================================================
# 14. VERIFY BINARY LABELS
# =============================================================================

unique_train_labels = torch.unique(Y_train).tolist()
unique_val_labels = torch.unique(Y_val).tolist()

print("\nTrain unique label values:", unique_train_labels)
print("Val unique label values  :", unique_val_labels)

assert set(unique_train_labels).issubset({0.0, 1.0})
assert set(unique_val_labels).issubset({0.0, 1.0})

print("\n✓ Targets are binary multi-label indicators.")


# =============================================================================
# 15. CLASS DISTRIBUTION
# =============================================================================

print("\n" + "-" * 80)
print("CLASS DISTRIBUTION")
print("-" * 80)

print(
    f"{'Disease':>8} | "
    f"{'Train +':>8} | "
    f"{'Train %':>8} | "
    f"{'Val +':>8} | "
    f"{'Val %':>8}"
)

print("-" * 55)

for i, disease in enumerate(DISEASE_NAMES):

    train_pos = int(Y_train[:, i].sum().item())
    val_pos = int(Y_val[:, i].sum().item())

    train_prev = train_pos / len(Y_train)
    val_prev = val_pos / len(Y_val)

    print(
        f"{disease:>8} | "
        f"{train_pos:8d} | "
        f"{train_prev:8.4f} | "
        f"{val_pos:8d} | "
        f"{val_prev:8.4f}"
    )


# =============================================================================
# 16. MULTI-LABEL CARDINALITY
# =============================================================================

train_cardinality = Y_train.sum(dim=1)
val_cardinality = Y_val.sum(dim=1)

print("\n" + "-" * 80)
print("MULTI-LABEL CARDINALITY")
print("-" * 80)

print(
    f"Train mean labels/patient : "
    f"{train_cardinality.mean().item():.4f}"
)

print(
    f"Val mean labels/patient   : "
    f"{val_cardinality.mean().item():.4f}"
)

print(
    f"Train max labels/patient  : "
    f"{int(train_cardinality.max().item())}"
)

print(
    f"Val max labels/patient    : "
    f"{int(val_cardinality.max().item())}"
)


# =============================================================================
# 17. VERIFY LOCKED REFERENCE
# =============================================================================

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E REFERENCE")
print("-" * 80)

locked_reference = fused_artifact[
    "locked_reference"
]

for key, value in locked_reference.items():
    print(
        f"{key:20s}: {value}"
    )

assert locked_reference["best_epoch"] == 5

assert abs(
    locked_reference["val_micro_f1"] - 0.535966
) < 1e-6

assert abs(
    locked_reference["val_macro_f1"] - 0.561936
) < 1e-6

assert abs(
    locked_reference["exact_match"] - 0.175799
) < 1e-6

print("\n✓ Locked Experiment E reference verified.")





CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS

Dataset root: /content/drive/My Drive/Eye Disease/Dataset
Fused artifact: stage2_experiment_e_fused_representations_seed42.pt

--------------------------------------------------------------------------------
LOADING LOCKED FUSED REPRESENTATIONS
--------------------------------------------------------------------------------
Artifact loaded successfully.
Artifact type: <class 'dict'>

--------------------------------------------------------------------------------
VERIFYING LOCKED PROVENANCE
--------------------------------------------------------------------------------
Experiment     : Experiment E — Strict Class-wise Adaptive Fusion
Experiment key : experiment_e
Seed           : 42
Status         : LOCKED

✓ Experiment E verified.
✓ Seed 42 verified.
✓ Artifact status = LOCKED.

--------------------------------------------------------------------------------
DISEASE ORDER
-----------------------------------------------------

###EXPERIMENT 1


UPSTREAM HANDOFF VERIFIED

Train:

    X_train : [2141, 8, 768]
    Y_train : [2141, 8]

Validation:

    X_val   : [438, 8, 768]
    Y_val   : [438, 8]

Representation:

    Eight disease-specific 768-D fused representations per patient.

Patient alignment:

    ✓ Verified
    ✓ Tensor IDs normalized to integer IDs

Labels:

    ✓ Binary multi-label
    ✓ Correct disease ordering

Experiment E:

    ✓ Locked
    ✓ Seed 42
    ✓ No upstream modification

IMPORTANT:

    No classifier has been trained yet.

The next step is architectural design of the multi-label classification
head using the locked disease-specific fused representations.



CELL 3 : REPRESENTATION-AWARE CLASSIFICATION HEAD


 Input:

       [B, 8, 768]

 Each of the 8 slots corresponds to one disease-specific representation.

 Architecture:

       Disease-specific fused features
                  [B, 8, 768]
                         │
                    LayerNorm
                         │
                Shared projection
                  768 → 512
                         │
                       ReLU
                         │
                     Dropout
                         │
                Disease-specific
                   scalar heads
                         │
                  [B, 8, 1]
                         │
                       logits

 The shared projection learns a common representation space while the
 disease-specific heads preserve independent classification decisions.


In [16]:
# =============================================================================
# QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION
# CELL 3 — REPRESENTATION-AWARE CLASSIFICATION HEAD
# =============================================================================

import torch
import torch.nn as nn

print("=" * 80)
print("CELL 3 — REPRESENTATION-AWARE MULTI-LABEL CLASSIFICATION HEAD")
print("=" * 80)


# =============================================================================
# 1. LOCKED INPUT CONFIGURATION
# =============================================================================

NUM_CLASSES = 8
INPUT_DIM = 768
HIDDEN_DIM = 512
DROPOUT = 0.30

print("\n" + "-" * 80)
print("LOCKED INPUT CONFIGURATION")
print("-" * 80)

print(f"Number of disease representations : {NUM_CLASSES}")
print(f"Representation dimension          : {INPUT_DIM}")
print(f"Hidden dimension                  : {HIDDEN_DIM}")
print(f"Dropout                           : {DROPOUT}")


# =============================================================================
# 2. REPRESENTATION-AWARE CLASSIFICATION HEAD
# =============================================================================

class RepresentationAwareMultiLabelHead(nn.Module):

    def __init__(
        self,
        input_dim=768,
        hidden_dim=512,
        num_classes=8,
        dropout=0.30
    ):

        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        # ---------------------------------------------------------------------
        # Per-disease normalization
        # ---------------------------------------------------------------------

        self.layer_norm = nn.LayerNorm(
            input_dim
        )

        # ---------------------------------------------------------------------
        # Shared representation projection
        # ---------------------------------------------------------------------

        self.shared_projection = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )

        # ---------------------------------------------------------------------
        # Independent disease classifiers
        #
        # Each disease receives its own classifier.
        # This is important because the eight diseases have very different
        # prevalence and visual characteristics.
        # ---------------------------------------------------------------------

        self.disease_heads = nn.ModuleList([

            nn.Linear(
                hidden_dim,
                1
            )

            for _ in range(num_classes)

        ])


    def forward(self, x):

        # Expected:
        # [batch, 8, 768]

        assert x.ndim == 3, (
            f"Expected [B, 8, 768], got {tuple(x.shape)}"
        )

        assert x.shape[-1] == self.input_dim, (
            f"Expected feature dimension {self.input_dim}, "
            f"got {x.shape[-1]}"
        )

        assert x.shape[1] == self.num_classes, (
            f"Expected {self.num_classes} disease representations, "
            f"got {x.shape[1]}"
        )

        # ---------------------------------------------------------------------
        # Layer normalization
        # ---------------------------------------------------------------------

        x = self.layer_norm(x)

        # ---------------------------------------------------------------------
        # Shared projection
        #
        # [B, 8, 768] → [B, 8, 512]
        # ---------------------------------------------------------------------

        x = self.shared_projection(x)

        # ---------------------------------------------------------------------
        # Independent disease heads
        # ---------------------------------------------------------------------

        logits = []

        for disease_idx, head in enumerate(self.disease_heads):

            disease_feature = x[:, disease_idx, :]

            disease_logit = head(
                disease_feature
            )

            logits.append(
                disease_logit
            )

        # [B, 8, 1] → [B, 8]

        logits = torch.cat(
            logits,
            dim=1
        )

        return logits


# =============================================================================
# 3. INSTANTIATE MODEL
# =============================================================================

model = RepresentationAwareMultiLabelHead(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
)

print("\n" + "-" * 80)
print("MODEL CREATED")
print("-" * 80)

print(model)


# =============================================================================
# 4. PARAMETER COUNT
# =============================================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "-" * 80)
print("PARAMETER COUNT")
print("-" * 80)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)


# =============================================================================
# 5. SHAPE TEST
# =============================================================================

print("\n" + "-" * 80)
print("FORWARD-PASS SHAPE TEST")
print("-" * 80)

model.eval()

with torch.no_grad():

    test_input = torch.randn(
        4,
        NUM_CLASSES,
        INPUT_DIM
    )

    test_output = model(
        test_input
    )

print(
    f"Input shape  : {tuple(test_input.shape)}"
)

print(
    f"Output shape : {tuple(test_output.shape)}"
)

assert test_output.shape == (
    4,
    NUM_CLASSES
)

print("\n✓ Forward pass successful.")
print("✓ Output contains 8 independent logits.")


# =============================================================================
# 6. VERIFY DISEASE-SLOT INDEPENDENCE
# =============================================================================

print("\n" + "-" * 80)
print("DISEASE-SLOT INDEPENDENCE TEST")
print("-" * 80)

# Change only disease slot 0.
# Other disease representations remain unchanged.

x_a = torch.randn(
    2,
    NUM_CLASSES,
    INPUT_DIM
)

x_b = x_a.clone()

x_b[:, 0, :] += 5.0

model.eval()

with torch.no_grad():

    logits_a = model(x_a)
    logits_b = model(x_b)

difference = (
    logits_a - logits_b
).abs()

print(
    "Mean absolute logit difference by disease:"
)

for i, disease in enumerate(DISEASE_NAMES):

    print(
        f"  {disease}: "
        f"{difference[:, i].mean().item():.6f}"
    )


# =============================================================================
# 7. LOGIT OUTPUT VERIFICATION
# =============================================================================

assert torch.isfinite(
    test_output
).all()

assert test_output.ndim == 2
assert test_output.shape[1] == NUM_CLASSES

print("\n✓ Logits are finite.")
print("✓ Output dimensionality = 8.")
print("✓ No softmax applied.")
print("✓ Classes remain independently represented.")


# =============================================================================
# 8. SIGMOID INTERPRETATION CHECK
# =============================================================================

with torch.no_grad():

    probabilities = torch.sigmoid(
        test_output
    )

assert torch.all(
    probabilities >= 0
)

assert torch.all(
    probabilities <= 1
)

print("\n" + "-" * 80)
print("MULTI-LABEL PROBABILITY CHECK")
print("-" * 80)

print(
    f"Probability range: "
    f"{probabilities.min().item():.6f} "
    f"→ "
    f"{probabilities.max().item():.6f}"
)

print("\n✓ Sigmoid produces independent [0,1] probabilities.")




CELL 3 — REPRESENTATION-AWARE MULTI-LABEL CLASSIFICATION HEAD

--------------------------------------------------------------------------------
LOCKED INPUT CONFIGURATION
--------------------------------------------------------------------------------
Number of disease representations : 8
Representation dimension          : 768
Hidden dimension                  : 512
Dropout                           : 0.3

--------------------------------------------------------------------------------
MODEL CREATED
--------------------------------------------------------------------------------
RepresentationAwareMultiLabelHead(
  (layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (shared_projection): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (disease_heads): ModuleList(
    (0-7): 8 x Linear(in_features=512, out_features=1, bias=True)
  )
)

------------------------------------------------

LOCKED UPSTREAM

Experiment E / Seed 42
Adaptive bilateral fusion
8 disease-specific representations
768 dimensions per representation
Upstream parameters FROZEN


CLASSIFICATION HEAD:

      Input                    : [B, 8, 768]
      Per-disease LayerNorm    : 768
      Shared projection        : 768 → 512
      Activation               : ReLU
      Dropout                  : 0.30

Disease-specific heads:

    N → 512 → 1
    D → 512 → 1
    G → 512 → 1
    C → 512 → 1
    A → 512 → 1
    H → 512 → 1
    M → 512 → 1
    O → 512 → 1

Output:

     [B, 8] logits
Probability              : Independent sigmoid

Loss                     : Asymmetric Loss (next cell)


DESIGN PRINCIPLE:

**The classifier preserves the disease-specific structure produced by
Experiment E instead of flattening all 8 × 768 features into one vector.
The shared projection controls parameter growth while the independent
heads allow each disease to learn its own decision boundary.**


CELL 4 — WEIGHTED RANDOM SAMPLING


OBJECTIVE

Address patient-level class imbalance during classifier training.

METHOD

WeightedRandomSampler is used for the training set.

Sampling weights are calculated from the patient's complete multi-label
target rather than treating each disease independently.

Rare disease labels receive greater sampling importance, while patients
containing multiple disease labels can contribute to multiple class
requirements.

Patients with no positive disease label retain a baseline sampling weight.

TRAINING:

    Batch size                 : 64
    Sampling                  : WeightedRandomSampler
    Replacement               : Yes
    Samples per epoch         : 2141

VALIDATION:

    Sampling                  : None
    Shuffle                   : No
    Distribution              : Natural validation distribution

This prevents the validation metrics from being artificially affected by
class-balancing during evaluation.

BALANCING STRATEGY

Patient-level balancing:
    WeightedRandomSampler

Example-level loss handling:
    Asymmetric Loss

These address different parts of the imbalance problem and will be evaluated
together during classifier training.

IMPORTANT

**No data is duplicated physically.
No validation patients are resampled.
No labels are modified.
No upstream Experiment E representation is modified.**



In [17]:
# =============================================================================
# CELL 4 — WEIGHTED RANDOM SAMPLING
# =============================================================================

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

BATCH_SIZE = 64

y = Y_train.float()

class_counts = y.sum(dim=0)
class_weights = y.shape[0] / (class_counts * NUM_CLASSES)

sample_weights = (y * class_weights).sum(dim=1)

# Give all-zero-label patients a normal baseline weight
zero_label = y.sum(dim=1) == 0
sample_weights[zero_label] = 1.0

sample_weights = sample_weights / sample_weights.mean()

sampler = WeightedRandomSampler(
    weights=sample_weights.double(),
    num_samples=len(sample_weights),
    replacement=True
)

train_dataset = TensorDataset(
    X_train.float(),
    Y_train.float()
)

val_dataset = TensorDataset(
    X_val.float(),
    Y_val.float()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Weighted sampling configured.")
print(f"Batch size: {BATCH_SIZE}")
print(f"Train samples per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Sample weight range: {sample_weights.min():.3f} - {sample_weights.max():.3f}")

assert len(train_dataset) == 2141
assert len(val_dataset) == 438
assert isinstance(train_loader.sampler, WeightedRandomSampler)
assert not isinstance(val_loader.sampler, WeightedRandomSampler)

print("✓ Train: WeightedRandomSampler")
print("✓ Validation: natural distribution")
print("✓ No validation resampling")

Weighted sampling configured.
Batch size: 64
Train samples per epoch: 34
Validation batches: 7
Sample weight range: 0.367 - 6.431
✓ Train: WeightedRandomSampler
✓ Validation: natural distribution
✓ No validation resampling


CELL 5 — INITIAL MULTI-LABEL CLASSIFIER TRAINING
OBJECTIVE

Train the first classifier using the locked Experiment E representations.

TRAINING:

    Optimizer: AdamW
    Learning rate: 0.001
    Weight decay: 0.0001
    Batch size: 64
    Maximum epochs: 40
    Early stopping patience: 8

IMBALANCE HANDLING:

    WeightedRandomSampler for training
    Asymmetric Loss for optimization
    Validation remains naturally distributed

MODEL SELECTION:

    Primary metric: Validation Macro-F1

    Also record:

    Micro-F1
    Validation loss
    Exact Match
    Per-disease F1
    THRESHOLD

    Initial threshold: 0.50

Validation probabilities will be retained for later threshold optimization without retraining.

**IMPORTANT**

*Experiment E, Seed 42 and all fused representations remain frozen. No upstream modification is performed.*

In [18]:
# =============================================================================
# CELL 5 — INITIAL MULTI-LABEL CLASSIFIER TRAINING
# =============================================================================

import copy
import numpy as np
import torch.nn as nn
from sklearn.metrics import f1_score


class AsymmetricLoss(nn.Module):

    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):

        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos

        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        xs_pos = xs_pos.clamp(self.eps, 1.0 - self.eps)
        xs_neg = xs_neg.clamp(self.eps, 1.0 - self.eps)

        loss = (
            targets * torch.log(xs_pos) +
            (1.0 - targets) * torch.log(xs_neg)
        )

        asymmetric_weight = (
            targets * (1.0 - xs_pos.detach()).pow(self.gamma_pos) +
            (1.0 - targets) * (1.0 - xs_neg.detach()).pow(self.gamma_neg)
        )

        return -(loss * asymmetric_weight).mean()


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8

model = RepresentationAwareMultiLabelHead(
    input_dim=768,
    hidden_dim=512,
    num_classes=8,
    dropout=0.30
).to(DEVICE)

criterion = AsymmetricLoss(
    gamma_neg=4.0,
    gamma_pos=1.0,
    clip=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

best_macro_f1 = -1.0
best_state = None
best_epoch = 0
history = []
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()

    val_loss = 0.0
    all_probs = []
    all_targets = []

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)

            all_probs.append(
                torch.sigmoid(logits).cpu().numpy()
            )

            all_targets.append(
                yb.cpu().numpy()
            )

    val_loss /= len(val_loader.dataset)

    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)

    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(
        targets, preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        targets, preds,
        average="macro",
        zero_division=0
    )

    exact_match = np.mean(
        np.all(preds == targets, axis=1)
    )

    per_class_f1 = f1_score(
        targets, preds,
        average=None,
        zero_division=0
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "exact_match": exact_match,
        "per_class_f1": per_class_f1.copy()
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"Micro {micro_f1:.4f} | "
        f"Macro {macro_f1:.4f} | "
        f"EM {exact_match:.4f}"
    )

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_probs = probs.copy()
        best_targets = targets.copy()
        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break


model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Macro-F1: {best_macro_f1:.4f}")

Epoch 01 | Train 0.0465 | Val 0.0586 | Micro 0.4759 | Macro 0.5069 | EM 0.0342
Epoch 02 | Train 0.0309 | Val 0.0618 | Micro 0.4964 | Macro 0.5329 | EM 0.0708
Epoch 03 | Train 0.0269 | Val 0.0615 | Micro 0.5006 | Macro 0.5315 | EM 0.0776
Epoch 04 | Train 0.0239 | Val 0.0673 | Micro 0.5041 | Macro 0.5355 | EM 0.0890
Epoch 05 | Train 0.0231 | Val 0.0670 | Micro 0.5227 | Macro 0.5424 | EM 0.1073
Epoch 06 | Train 0.0216 | Val 0.0670 | Micro 0.5292 | Macro 0.5455 | EM 0.1027
Epoch 07 | Train 0.0190 | Val 0.0711 | Micro 0.5213 | Macro 0.5564 | EM 0.0959
Epoch 08 | Train 0.0185 | Val 0.0697 | Micro 0.5286 | Macro 0.5427 | EM 0.1187
Epoch 09 | Train 0.0173 | Val 0.0682 | Micro 0.5370 | Macro 0.5438 | EM 0.1393
Epoch 10 | Train 0.0181 | Val 0.0696 | Micro 0.5407 | Macro 0.5583 | EM 0.1393
Epoch 11 | Train 0.0161 | Val 0.0765 | Micro 0.5407 | Macro 0.5604 | EM 0.1347
Epoch 12 | Train 0.0151 | Val 0.0806 | Micro 0.5344 | Macro 0.5598 | EM 0.1233
Epoch 13 | Train 0.0141 | Val 0.0823 | Micro 0.5366 

###EXPERIMENT 2

CELL 6 — EXPERIMENT F CLASSIFIER ARCHITECTURE


CONFIGURATION

    Experiment              : 2
    Upstream representation : Experiment E / Seed 42
    Upstream fusion         : FROZEN

ARCHITECTURAL CHANGES

    1. LayerNorm on the 768-D disease-specific representations.

    2. Double-dropout bottleneck:
          LayerNorm
              ↓
          Dropout(0.3)
              ↓
          FC(768 → 512)
              ↓
          ReLU
              ↓
          Dropout(0.3)

    3. Global disease-context pathway:
          Mean over 8 disease representations
              ↓
          FC(768 → 512)
              ↓
          Context added to every disease-specific 512-D representation.

    4. Eight independent classification outputs.


OUTPUT

Input  : [B, 8, 768]

Output : [B, 8]


Each output corresponds independently to:
    *N, D, G, C, A, H, M, O*


TRAINING CHANGES FOR EXPERIMENT 2

    Sampling              : WeightedRandomSampler
    ASL gamma_neg         : 2.0
    ASL gamma_pos         : 1.0
    ASL clipping          : 0.05
    AdamW weight decay    : 1e-2

The locked Experiment E fused representations are not modified.



In [19]:
# =============================================================================
# CELL 6 — EXPERIMENT 2 CLASSIFIER ARCHITECTURE
# =============================================================================

import torch
import torch.nn as nn


class RepresentationAwareMultiLabelHeadF(nn.Module):

    def __init__(self, input_dim=768, hidden_dim=512, num_classes=8, dropout=0.30):
        super().__init__()

        self.norm = nn.LayerNorm(input_dim)
        self.input_dropout = nn.Dropout(dropout)

        self.shared_projection = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.hidden_dropout = nn.Dropout(dropout)

        self.context_projection = nn.Linear(input_dim, hidden_dim)

        self.disease_heads = nn.ModuleList([
            nn.Linear(hidden_dim, 1)
            for _ in range(num_classes)
        ])

    def forward(self, x):

        x = self.norm(x)
        x = self.input_dropout(x)

        disease_features = self.shared_projection(x)
        disease_features = self.activation(disease_features)
        disease_features = self.hidden_dropout(disease_features)

        global_context = x.mean(dim=1)
        global_context = self.context_projection(global_context)
        global_context = global_context.unsqueeze(1)

        disease_features = disease_features + global_context

        logits = torch.cat(
            [head(disease_features[:, i, :])
             for i, head in enumerate(self.disease_heads)],
            dim=1
        )

        return logits


model_2 = RepresentationAwareMultiLabelHeadF(
    input_dim=768,
    hidden_dim=512,
    num_classes=8,
    dropout=0.30
).to(DEVICE)

param_count = sum(
    p.numel() for p in model_2.parameters()
    if p.requires_grad
)

test_logits = model_2(X_train[:4].float().to(DEVICE))

print(f"Experiment F parameters: {param_count:,}")
print(f"Test output shape: {tuple(test_logits.shape)}")

assert test_logits.shape == (4, 8)
assert param_count > 0

print("✓ Double-dropout bottleneck verified")
print("✓ Global disease-context pathway verified")
print("✓ Eight independent outputs verified")
print("✓ Experiment F architecture ready")

Experiment F parameters: 793,096
Test output shape: (4, 8)
✓ Double-dropout bottleneck verified
✓ Global disease-context pathway verified
✓ Eight independent outputs verified
✓ Experiment F architecture ready
